In [19]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

In [20]:
df = pd.read_csv('/content/drive/MyDrive/DS & ML/diabetic_data.csv')

In [21]:
df.head(10)

,encounter_id,patient_nbr,race,gender,age,weight,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,...,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted
0,2278392,8222157,Caucasian,Female,[0-10),?,6,25,1,1,...,No,No,No,No,No,No,No,No,No,NO
1,149190,55629189,Caucasian,Female,[10-20),?,1,1,7,3,...,No,Up,No,No,No,No,No,Ch,Yes,>30
2,64410,86047875,AfricanAmerican,Female,[20-30),?,1,1,7,2,...,No,No,No,No,No,No,No,No,Yes,NO
3,500364,82442376,Caucasian,Male,[30-40),?,1,1,7,2,...,No,Up,No,No,No,No,No,Ch,Yes,NO
4,16680,42519267,Caucasian,Male,[40-50),?,1,1,7,1,...,No,Steady,No,No,No,No,No,Ch,Yes,NO
5,35754,82637451,Caucasian,Male,[50-60),?,2,1,2,3,...,No,Steady,No,No,No,No,No,No,Yes,>30
6,55842,84259809,Caucasian,Male,[60-70),?,3,1,2,4,...,No,Steady,No,No,No,No,No,Ch,Yes,NO
7,63768,114882984,Caucasian,Male,[70-80),?,1,1,7,5,...,No,No,No,No,No,No,No,No,Yes,>30
8,12522,48330783,Caucasian,Female,[80-90),?,2,1,4,13,...,No,Steady,No,No,No,No,No,Ch,Yes,NO
9,15738,63555939,Caucasian,Female,[90-100),?,3,3,4,12,...,No,Steady,No,No,No,No,No,Ch,Yes,NO


In [22]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 101766 entries, 0 to 101765
Data columns (total 50 columns):
 #   Column                    Non-Null Count   Dtype 
---  ------                    --------------   ----- 
 0   encounter_id              101766 non-null  int64 
 1   patient_nbr               101766 non-null  int64 
 2   race                      101766 non-null  object
 3   gender                    101766 non-null  object
 4   age                       101766 non-null  object
 5   weight                    101766 non-null  object
 6   admission_type_id         101766 non-null  int64 
 7   discharge_disposition_id  101766 non-null  int64 
 8   admission_source_id       101766 non-null  int64 
 9   time_in_hospital          101766 non-null  int64 
 10  payer_code                101766 non-null  object
 11  medical_specialty         101766 non-null  object
 12  num_lab_procedures        101766 non-null  int64 
 13  num_procedures            101766 non-null  int64 
 14  num_

In [23]:
df.shape

(101766, 50)

In [24]:
# replacing invalid values
df.replace(('?','Unknown/Invalid'), np.nan, inplace=True)

# Check for missing values
missing_values = df.isnull().sum()
print(missing_values)

encounter_id                    0
patient_nbr                     0
race                         2273
gender                          3
age                             0
weight                      98569
admission_type_id               0
discharge_disposition_id        0
admission_source_id             0
time_in_hospital                0
payer_code                  40256
medical_specialty           49949
num_lab_procedures              0
num_procedures                  0
num_medications                 0
number_outpatient               0
number_emergency                0
number_inpatient                0
diag_1                         21
diag_2                        358
diag_3                       1423
number_diagnoses                0
max_glu_serum               96420
A1Cresult                   84748
metformin                       0
repaglinide                     0
nateglinide                     0
chlorpropamide                  0
glimepiride                     0
acetohexamide 

## **DATA CLEANING**
Data imputation strategies to fill missing values were tailored to variable types:

* categorical variables are imputed using the mode to preserve category distribution
* numerical variables are imputed using the median to mitigate the effect of outliers.
* Columns with excessive missing values are dropped to maintain data quality and model reliability.


In [25]:
# remove duplicates in patient encounter; keep only first encounter per patient
df = df.sort_values('encounter_id').drop_duplicates(subset='patient_nbr', keep='first')
print(f"Shape after dedup (one row per patient): {df.shape}")

Shape after dedup (one row per patient): (71518, 50)


In [26]:
# Dropping irrelevant columns
df.drop(['encounter_id', 'patient_nbr'], axis=1, inplace=True, errors='ignore')

# Dropping columns with high missing values
df.drop(['weight', 'max_glu_serum', 'A1Cresult'], axis=1, inplace=True, errors='ignore')

# Filling categorical columns with mode
cat_cols = df.select_dtypes(include='object').columns
for col in cat_cols:
    df[col].fillna(df[col].mode()[0], inplace=True)

# Filling numerical columns with median
num_cols = df.select_dtypes(include=['float64', 'int64']).columns
for col in num_cols:
    df[col].fillna(df[col].median(), inplace=True)

In [27]:
# Mapping IDs to readable labels
admission_type_map = {
    1:'Emergency', 2:'Urgent', 3:'Elective',
    4:'Newborn', 5:'Not Available', 6:'NULL',
    7:'Trauma Center', 8:'Not Mapped'
}
discharge_map = {
    1:'Discharged to home', 2:'Transferred-short term hospital',
    3:'Transferred-SNF', 4:'Transferred-ICF',
    5:'Transferred-inpatient care', 6:'Home with health service',
    7:'Left AMA', 8:'Home-IV provider', 9:'Admitted inpatient',
    10:'Neonate-neonatal aftercare', 11:'Expired',
    12:'Still patient', 13:'Hospice/home', 14:'Hospice/facility',
    15:'Discharged/transferred within institution',
    16:'Discharged/transferred to another institution for outpatient services',
    17:'Discharged/transferred within institution for outpatient services',
    18:'NULL',
    19: 'Expired at home',
    20: 'Expired in a medical facility',
    21: 'Expired in an unknown place',
    22: 'Discharged/transferred to another rehab facility',
    23: 'Discharged/transferred to a long term care hospital',
    24: 'Discharged/transferred to a nursing facility',
    25:'Not Mapped', 26:'Unknown/Invalid', 27: 'Discharged/transferred to a federal healthcare facility',
    28:'Discharged/transferred to a psychiatric hospital of psychiatric distinct part of a hospital',
    29:'Discharged/transferred to a Critical Access Hospital (CAH)',
    30: 'Discharged/transferred to another type of health care facility'
}
admission_source_map = {
    1:'Physician Referral', 2:'Clinic Referral', 3:'HMO Referral',
    4:'Transfer-hospital', 5:'Transfer-SNF', 6:'Transfer-medical facility',
    7:'Emergency Room', 8:'Court/Law Enforcement', 9:'Not Available',
    10:'Transfer-CAH', 11:'Normal Delivery', 12:'Premature Delivery',
    13:'Sick Baby', 14:'Extramural Birth', 15:'Not Available', 17:'NULL',
    18:'Transfer-Another Home Health Agency', 19:'Readmission-Same Home Health Agency',
    20:'Not Mapped', 21:'Unknown/Invalid', 22:'Transfer-Hospital(sep claim)',
    23:'Born inside this hospital', 24:'Born outside this hospital',
    25:'Transfer-Ambulatory Surgery Center', 26:'Transfer-Hospice'
}

df['admission_type_id']         = df['admission_type_id'].map(admission_type_map)
df['discharge_disposition_id']  = df['discharge_disposition_id'].map(discharge_map)
df['admission_source_id']       = df['admission_source_id'].map(admission_source_map)

print("Mapped admission_type_id:", df['admission_type_id'].unique())

Mapped admission_type_id: ['Urgent' 'Elective' 'Emergency' 'NULL' 'Newborn' 'Not Available'
 'Not Mapped' 'Trauma Center']


In [28]:
# Remove records where discharge disposition indicates death/hospice
# (patient cannot be readmitted)
exclude_discharge = ['Expired', 'Hospice/home', 'Hospice/facility']
df = df[~df['discharge_disposition_id'].isin(exclude_discharge)]
print(f"Shape after removing hospice/expired: {df.shape}")

Shape after removing hospice/expired: (69980, 45)


In [29]:
# replacing values
df.replace(('NULL','Unknown/Invalid','Not Mapped','Not Available'), 'Unknown', inplace=True)

In [30]:
# Verify absence of missing values
missing_values = df.isnull().sum()
print(missing_values)

race                        0
gender                      0
age                         0
admission_type_id           0
discharge_disposition_id    0
admission_source_id         0
time_in_hospital            0
payer_code                  0
medical_specialty           0
num_lab_procedures          0
num_procedures              0
num_medications             0
number_outpatient           0
number_emergency            0
number_inpatient            0
diag_1                      0
diag_2                      0
diag_3                      0
number_diagnoses            0
metformin                   0
repaglinide                 0
nateglinide                 0
chlorpropamide              0
glimepiride                 0
acetohexamide               0
glipizide                   0
glyburide                   0
tolbutamide                 0
pioglitazone                0
rosiglitazone               0
acarbose                    0
miglitol                    0
troglitazone                0
tolazamide

## Feature Engineering

In [31]:
# converting the target class to binary classification
df['readmitted'] = df['readmitted'].replace({'<30': 1, '>30': 0, 'NO': 0})
display(df['readmitted'].head())

,readmitted
8,0
9,0
4,0
10,0
5,0


In [32]:
# converting the grouped age column to midrange
df['age'] = df['age'].str.replace(r'[[()]', '', regex=True)
df[['age_min', 'age_max']] = df['age'].str.split('-', expand=True)
df['age'] = (df['age_min'].astype(int) + df['age_max'].astype(int)) / 2
df.drop(['age_min', 'age_max'], axis=1, inplace=True, errors='ignore')
display(df['age'].head())

,age
8,85.0
9,95.0
4,45.0
10,45.0
5,55.0


In [33]:
# create a column for total visits
df['total_visits'] = df['number_outpatient'] + df['number_emergency'] + df['number_inpatient']
df.drop(['number_outpatient', 'number_emergency', 'number_inpatient'], axis=1, inplace=True, errors='ignore')
display(df['total_visits'].head())

,total_visits
8,0
9,0
4,0
10,0
5,0


In [34]:
# converting the medication columns to binary to engineer a total column
medication_columns = [
    'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide',
    'glimepiride', 'acetohexamide', 'glipizide', 'glyburide', 'tolbutamide',
    'pioglitazone', 'rosiglitazone', 'acarbose', 'miglitol', 'troglitazone',
    'tolazamide', 'examide', 'citoglipton', 'insulin', 'glyburide-metformin',
    'glipizide-metformin', 'glimepiride-pioglitazone', 'metformin-rosiglitazone',
    'metformin-pioglitazone'
]

for col in medication_columns:
    df[col] = df[col].apply(lambda x: 0 if x == 'No' else 1).astype(int)


# Sum the binary medication columns to get the total number of distinct medication types
df['total_med_types'] = df[medication_columns].sum(axis=1)

print(df[['num_medications', 'total_med_types']])

        num_medications  total_med_types
8                    28                2
9                    18                2
4                     8                2
10                   17                1
5                    16                1
...                 ...              ...
101754               33                2
101755               26                2
101756               17                1
101758               22                1
101765                3                0

[69980 rows x 2 columns]


* The `num_medications` column which already existed in the dataset, represents the total count of all medications prescribed.
* The newly created `total_med_types` column represents the count of distinct types of medications (e.g., if a patient is on Metformin and Insulin, total_med_types would be 2, regardless of the specific dosage or frequency).

In [35]:
df.tail(5)

,race,gender,age,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,payer_code,medical_specialty,num_lab_procedures,...,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted,total_visits,total_med_types
101754,Caucasian,Female,75.0,Emergency,Discharged to home,Emergency Room,9,MC,InternalMedicine,50,...,0,0,0,0,0,Ch,Yes,0,0,2
101755,Other,Female,45.0,Emergency,Discharged to home,Emergency Room,14,MD,InternalMedicine,73,...,0,0,0,0,0,Ch,Yes,0,1,2
101756,Other,Female,65.0,Emergency,Discharged to home,Emergency Room,2,MD,InternalMedicine,46,...,0,0,0,0,0,No,Yes,0,3,1
101758,Caucasian,Female,85.0,Emergency,Discharged to home,Emergency Room,5,MC,InternalMedicine,76,...,0,0,0,0,0,Ch,Yes,0,1,1
101765,Caucasian,Male,75.0,Emergency,Discharged to home,Emergency Room,6,MC,InternalMedicine,13,...,0,0,0,0,0,No,No,0,0,0


In [36]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 69980 entries, 8 to 101765
Data columns (total 44 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   race                      69980 non-null  object 
 1   gender                    69980 non-null  object 
 2   age                       69980 non-null  float64
 3   admission_type_id         69980 non-null  object 
 4   discharge_disposition_id  69980 non-null  object 
 5   admission_source_id       69980 non-null  object 
 6   time_in_hospital          69980 non-null  int64  
 7   payer_code                69980 non-null  object 
 8   medical_specialty         69980 non-null  object 
 9   num_lab_procedures        69980 non-null  int64  
 10  num_procedures            69980 non-null  int64  
 11  num_medications           69980 non-null  int64  
 12  diag_1                    69980 non-null  object 
 13  diag_2                    69980 non-null  object 
 14  diag_3    

In [37]:
df.shape

(69980, 44)

In [38]:
# Save the cleaned dataframe to your processed folder
df.to_csv('/content/drive/MyDrive/DS & ML/diabetic_data_clean.csv', index=False)

print("Done")


Done
